# Khử trùng lặp 3 cấp độ — Google Colab

Notebook này gộp đầy đủ ba cấp độ trong tài liệu XU_LY_COMMENT_TRUNG_LAP.md:

1. Cấp 1 — trùng hoàn toàn: comment giống 100% ký tự, giữ dòng đầu tiên.
2. Cấp 2 — trùng sau chuẩn hóa: so sánh chữ thường, bỏ dấu câu/emoji và
   chuẩn hóa khoảng trắng, nhưng vẫn giữ nguyên comment gốc.
3. Cấp 3 — trùng ngữ nghĩa: embedding tiếng Việt + cosine similarity.

Đầu vào là comments_cleaned_to_5_4.csv từ notebook làm sạch. Nếu đã lọc
tiếng Anh, dùng comments_cleaned_to_5_4_no_english.csv thay thế.

Cấp 1–2 được áp dụng tự động vì quy tắc rõ ràng. Cấp 3 mặc định chỉ tạo
candidate để review; không tự xóa câu chỉ vì chúng gần nghĩa nhau.


In [ ]:
# Cài thư viện cho Google Colab

!pip -q install sentence-transformers faiss-cpu pandas numpy


## 1. Tải dữ liệu sau làm sạch lên Colab

Chạy cell dưới và chọn một trong hai file:

- comments_cleaned_to_5_4.csv;
- comments_cleaned_to_5_4_no_english.csv nếu đã chạy notebook lọc tiếng Anh.

Nếu dùng Google Drive, có thể thay cell upload bằng drive.mount() và gán
INPUT_PATH tới file trong Drive.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded_files = files.upload()
INPUT_PATH = Path(next(iter(uploaded_files)))

print(f'Đã chọn file: {INPUT_PATH.name}')


In [ ]:
import re
import unicodedata
import json
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Đọc cấu hình nếu đã tải preprocessing_config.json lên Colab.
DEFAULT_CONFIG = {
    'semantic_model': 'bkai-foundation-models/vietnamese-bi-encoder',
    'semantic_similarity_threshold': 0.95,
    'semantic_top_k_neighbors': 10,
    'apply_semantic_removal': False,
    'random_seed': 42,
}
CONFIG_PATH = Path('preprocessing_config.json')
if CONFIG_PATH.exists():
    DEFAULT_CONFIG.update(json.loads(CONFIG_PATH.read_text(encoding='utf-8')))

MODEL_NAME = DEFAULT_CONFIG['semantic_model']
SIMILARITY_THRESHOLD = DEFAULT_CONFIG['semantic_similarity_threshold']
TOP_K_NEIGHBORS = DEFAULT_CONFIG['semantic_top_k_neighbors']
APPLY_AUTO_REMOVAL = DEFAULT_CONFIG['apply_semantic_removal']
RANDOM_SEED = DEFAULT_CONFIG['random_seed']
print(f'Model: {MODEL_NAME}; threshold: {SIMILARITY_THRESHOLD}')


In [ ]:
# Đọc và kiểm tra dữ liệu đầu vào

comments_before_deduplication = pd.read_csv(
    INPUT_PATH,
    encoding='utf-8-sig',
    dtype='string',
    keep_default_na=False,
)

required_columns = {'row_id', 'comment'}
missing_columns = required_columns - set(comments_before_deduplication.columns)
if missing_columns:
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(missing_columns)}')

comments_before_deduplication = comments_before_deduplication[
    ['row_id', 'comment']
].copy()
comments_before_deduplication['comment'] = (
    comments_before_deduplication['comment'].fillna('').astype('string').str.strip()
)
comments_before_deduplication = comments_before_deduplication.loc[
    comments_before_deduplication['comment'].ne('')
].reset_index(drop=True)

print(f'Số comment trước khử trùng lặp: {len(comments_before_deduplication):,}')


## 2. Cấp 1 và 2 — trùng hoàn toàn, trùng sau chuẩn hóa

Hai cấp đầu giữ dòng xuất hiện đầu tiên. Các dòng loại ra được lưu thành bảng
audit để có thể kiểm tra lại row_id và lý do loại.


In [ ]:
# Hàm khử trùng lặp cấp 1 và cấp 2

def build_normalized_duplicate_key(text):
    """Tạo khóa: chữ thường, bỏ dấu câu/emoji, khoảng trắng thống nhất."""
    text = unicodedata.normalize('NFC', str(text)).casefold()
    text = re.sub(r'[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def remove_exact_duplicates(data):
    """Giữ dòng đầu tiên khi comment giống hoàn toàn từng ký tự."""
    audit = data[['row_id', 'comment']].copy()
    audit['exact_group_size'] = audit.groupby('comment')['row_id'].transform('size')
    audit['is_exact_duplicate'] = audit.duplicated('comment', keep='first')
    kept = data.loc[~audit['is_exact_duplicate']].copy()
    removed = audit.loc[audit['is_exact_duplicate']].copy()
    return kept, removed, audit


def remove_normalized_duplicates(data):
    """Giữ dòng đầu tiên khi khóa chuẩn hóa trùng nhau; không sửa comment gốc."""
    audit = data[['row_id', 'comment']].copy()
    audit['normalized_key'] = audit['comment'].map(build_normalized_duplicate_key)
    audit['normalized_group_size'] = audit.groupby('normalized_key')['row_id'].transform('size')
    audit['is_normalized_duplicate'] = (
        audit['normalized_key'].ne('')
        & audit.duplicated('normalized_key', keep='first')
    )
    kept = data.loc[~audit['is_normalized_duplicate']].copy()
    removed = audit.loc[audit['is_normalized_duplicate']].copy()
    return kept, removed, audit


In [ ]:
# Áp dụng cấp 1 và cấp 2

comments_after_exact, removed_exact, exact_audit = remove_exact_duplicates(
    comments_before_deduplication
)
comments_after_normalized, removed_normalized, normalized_audit = (
    remove_normalized_duplicates(comments_after_exact)
)
comments = comments_after_normalized[['row_id', 'comment']].reset_index(drop=True)

deduplication_summary = pd.DataFrame({
    'metric': [
        'before_deduplication',
        'removed_exact_duplicates',
        'after_exact_deduplication',
        'removed_normalized_duplicates',
        'comments_without_normalized_key',
        'after_level_2_deduplication',
    ],
    'records': [
        len(comments_before_deduplication),
        len(removed_exact),
        len(comments_after_exact),
        len(removed_normalized),
        int(normalized_audit['normalized_key'].eq('').sum()),
        len(comments),
    ],
})

display(deduplication_summary)
display(removed_exact.head(10))
display(removed_normalized.head(10))
print(f'Số comment đưa vào cấp 3: {len(comments):,}')

# Tạo bản đồ nhóm để chia train/test mà không bị leakage
deduplication_group_map = comments_before_deduplication[['row_id', 'comment']].copy()
deduplication_group_map['exact_group_id'] = deduplication_group_map.groupby('comment', sort=False).ngroup()
deduplication_group_map['normalized_key'] = deduplication_group_map['comment'].map(build_normalized_duplicate_key)
deduplication_group_map['normalized_group_id'] = deduplication_group_map.groupby('normalized_key', sort=False).ngroup()
deduplication_group_map['exact_representative_row_id'] = deduplication_group_map.groupby('exact_group_id', sort=False)['row_id'].transform('first')
deduplication_group_map['normalized_representative_row_id'] = deduplication_group_map.groupby('normalized_group_id', sort=False)['row_id'].transform('first')


## 3. Cấp 3 — tìm candidate trùng ngữ nghĩa

Cấp 3 chỉ tạo candidate và bảng đề xuất loại. Hãy xem nội dung hai comment
trước khi đổi APPLY_AUTO_REMOVAL = True.


In [ ]:
# Tạo embedding tiếng Việt và chuẩn hóa vector để inner product = cosine similarity

model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    comments['comment'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
embeddings = np.ascontiguousarray(embeddings.astype('float32'))

print(f'Kích thước embedding: {embeddings.shape}')


In [ ]:
# Tạo chỉ mục HNSW và tìm các hàng xóm ngữ nghĩa gần nhất

def find_semantic_candidates(vectors, threshold, top_k):
    """Trả về các cặp có cosine similarity đạt ngưỡng.

    HNSW nhanh hơn so sánh toàn bộ mọi cặp, phù hợp với tập hơn 50.000 comment.
    """
    if len(vectors) < 2:
        return pd.DataFrame(columns=['source_position', 'candidate_position', 'cosine_similarity'])

    vector_dimension = vectors.shape[1]
    index = faiss.IndexHNSWFlat(vector_dimension, 32, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = 200
    index.hnsw.efSearch = 128
    index.add(vectors)

    scores, neighbor_positions = index.search(vectors, top_k + 1)
    candidate_rows = []

    for source_position, (row_scores, row_neighbors) in enumerate(
        zip(scores, neighbor_positions)
    ):
        for similarity, candidate_position in zip(row_scores, row_neighbors):
            if candidate_position <= source_position:
                continue
            if similarity < threshold:
                continue
            candidate_rows.append({
                'source_position': source_position,
                'candidate_position': int(candidate_position),
                'cosine_similarity': float(similarity),
            })

    return pd.DataFrame(
        candidate_rows,
        columns=['source_position', 'candidate_position', 'cosine_similarity'],
    )

semantic_candidates = find_semantic_candidates(
    embeddings,
    SIMILARITY_THRESHOLD,
    TOP_K_NEIGHBORS,
)

print(f'Số cặp candidate có cosine >= {SIMILARITY_THRESHOLD}: {len(semantic_candidates):,}')


In [ ]:
# Hiển thị candidate để kiểm tra trước khi xóa

def add_comment_text(candidate_data, data):
    """Gắn row_id và nội dung comment vào mỗi cặp candidate."""
    result = candidate_data.copy()
    result['source_row_id'] = result['source_position'].map(data['row_id'])
    result['candidate_row_id'] = result['candidate_position'].map(data['row_id'])
    result['source_comment'] = result['source_position'].map(data['comment'])
    result['candidate_comment'] = result['candidate_position'].map(data['comment'])
    return result[
        [
            'source_row_id', 'candidate_row_id', 'cosine_similarity',
            'source_comment', 'candidate_comment',
        ]
    ].sort_values('cosine_similarity', ascending=False)

candidate_review = add_comment_text(semantic_candidates, comments)
from google.colab import data_table

data_table.DataTable(
    candidate_review,
    include_index=False,
    num_rows_per_page=50,
)


In [ ]:
# Chọn comment cần xóa theo cách an toàn, không tạo chuỗi xóa bắc cầu

def choose_semantic_duplicates(candidate_data):
    """Giữ comment xuất hiện trước, chỉ xóa một candidate khi cả hai còn được giữ.

    Cách greedy này tránh việc A gần B và B gần C làm C bị xóa dù A không gần C.
    """
    removed_positions = set()
    decisions = []

    ordered_pairs = candidate_data.sort_values(
        'cosine_similarity',
        ascending=False,
    )

    for pair in ordered_pairs.itertuples(index=False):
        if pair.source_position in removed_positions:
            continue
        if pair.candidate_position in removed_positions:
            continue

        removed_positions.add(pair.candidate_position)
        decisions.append({
            'kept_position': pair.source_position,
            'removed_position': pair.candidate_position,
            'cosine_similarity': pair.cosine_similarity,
        })

    return pd.DataFrame(
        decisions,
        columns=['kept_position', 'removed_position', 'cosine_similarity'],
    )

semantic_removal_decisions = choose_semantic_duplicates(semantic_candidates)
semantic_removal_review = add_comment_text(
    semantic_removal_decisions.rename(
        columns={
            'kept_position': 'source_position',
            'removed_position': 'candidate_position',
        }
    ),
    comments,
)

from google.colab import data_table

data_table.DataTable(
    semantic_removal_review,
    include_index=False,
    num_rows_per_page=50,
)
print(f'Số comment có thể xóa sau review: {len(semantic_removal_review):,}')


In [ ]:
# Xuất audit cấp 1–2, candidate cấp 3 và dataset

OUTPUT_DIR = Path('/content/deduplication_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Thư mục output: {OUTPUT_DIR.resolve()}')

removed_exact.to_csv(
    OUTPUT_DIR / 'exact_duplicates_removed.csv',
    index=False,
    encoding='utf-8-sig',
)
removed_normalized.to_csv(
    OUTPUT_DIR / 'normalized_duplicates_removed.csv',
    index=False,
    encoding='utf-8-sig',
)
deduplication_group_map.to_csv(
    OUTPUT_DIR / 'deduplication_group_map.csv',
    index=False,
    encoding='utf-8-sig',
)
candidate_review.to_csv(
    OUTPUT_DIR / 'semantic_duplicate_candidates.csv',
    index=False,
    encoding='utf-8-sig',
)
semantic_removal_review.to_csv(
    OUTPUT_DIR / 'semantic_removal_review.csv',
    index=False,
    encoding='utf-8-sig',
)

# Output chắc chắn: kết quả sau cả cấp 1 và cấp 2.
comments.to_csv(
    OUTPUT_DIR / 'comments_cleaned_to_5_6.csv',
    index=False,
    encoding='utf-8-sig',
)
print(f'Đã xuất comments_cleaned_to_5_6.csv: {len(comments):,} dòng')

if APPLY_AUTO_REMOVAL:
    positions_to_remove = set(semantic_removal_decisions['removed_position'])
    semantic_deduplicated_comments = comments.drop(
        index=list(positions_to_remove)
    ).reset_index(drop=True)
    semantic_deduplicated_comments.to_csv(
        OUTPUT_DIR / 'comments_cleaned_to_semantic.csv',
        index=False,
        encoding='utf-8-sig',
    )
    print(
        'Đã xuất comments_cleaned_to_semantic.csv: '
        f'{len(semantic_deduplicated_comments):,} dòng'
    )
else:
    print('Cấp 3 chưa xóa tự động. Hãy review semantic_removal_review.csv trước.')

print('Đã xuất audit cấp 1–2 và hai bảng review cấp 3.')
print('Hãy chạy cell kế tiếp để tải semantic_removal_review.csv về máy.')


In [ ]:
# Tải file review về máy

from google.colab import files

review_path = OUTPUT_DIR / 'semantic_removal_review.csv'
print(f'File review: {review_path.resolve()}')
print(f'Số dòng review: {len(semantic_removal_review):,}')
files.download(str(review_path))
